## 1. 核心思想
单一的向量检索（稠密检索）可能遗漏关键词精确匹配的结果。

混合检索**同时使用多种检索方法**（如向量检索 + BM25），

然后通过 **Reranker** 对多路结果进行重新排序，取长补短，提升召回质量。

## 2. 完整代码实现 (Python + Milvus + BGE-M3 + BGE-Reranker)
我们需要一个支持稀疏向量 (Sparse Vector) 的模型。这里推荐使用 BGE-M3，它同时生成稠密向量和稀疏向量。重排序模型使用 BGE-Reranker-v2-m3。

### 3.1 配置 (初始化模型与客户端)

In [ ]:
import time
from pymilvus import (
    MilvusClient,
    DataType,
    AnnSearchRequest,
    RRFRanker
)
# 核心：导入 pymilvus 官方模型封装
from pymilvus.model.hybrid import BGEM3EmbeddingFunction
from pymilvus.model.reranker import BGERerankFunction

# --- 1. 初始化模型与客户端 ---

# 模型配置 (根据你的硬件调整)
DEVICE = "cpu"       # 如果有显卡改成 "cuda:0"
USE_FP16 = False     # CPU 必须 False，GPU 建议 True

print(">>> 正在加载 BGE-M3 模型...")
# 自动处理 Dense(1024维) + Sparse(词权重)
bge_m3_ef = BGEM3EmbeddingFunction(
    model_name='BAAI/bge-m3',
    device=DEVICE,
    use_fp16=USE_FP16
)

print(">>> 正在加载 BGE-Reranker 模型...")
# 自动处理 Cross-Encoder 打分
bge_reranker = BGERerankFunction(
    model_name="BAAI/bge-reranker-v2-m3",
    device=DEVICE,
    use_fp16=USE_FP16
)

# 连接 Milvus
milvus_client = MilvusClient(uri="http://111.228.53.183:19530")
COLLECTION_NAME = "hybrid_model_demo"

## 3.2 创建集合(collection)、schema、index

In [ ]:
if milvus_client.has_collection(COLLECTION_NAME):
    milvus_client.drop_collection(COLLECTION_NAME)

# 2.1 定义 Schema
schema = milvus_client.create_schema(auto_id=True, enable_dynamic_field=True)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=65535)
# 稠密向量 (BGE-M3 默认 1024)
schema.add_field(field_name="dense_vector", datatype=DataType.FLOAT_VECTOR, dim=1024)
# 稀疏向量
schema.add_field(field_name="sparse_vector", datatype=DataType.SPARSE_FLOAT_VECTOR)

# 2.2 定义索引
index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="dense_vector",
    index_type="HNSW",
    metric_type="COSINE",
    params={"M": 16, "efConstruction": 64}
)

index_params.add_index(
    field_name="sparse_vector",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="IP", # 必须要用IP “我给你的这个 0.217 就是最终结论，你别动它，直接存进倒排索引里。
    params={"drop_ratio_build": 0.2}
)

# 2.3 创建
milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema,
    index_params=index_params
)
print(f"\n>>> 集合 {COLLECTION_NAME} 创建成功。")

### 3.3 插入数据

In [ ]:
# --- 3. 插入数据 (使用 SDK 自动向量化) ---

documents = [
    "错误代码 0x80040115 表示 Outlook 连接 Exchange 服务器失败。请检查网络设置。",
    "我的邮件发不出去了，好像连不上网，怎么解决？",
    "Exchange 服务器的连接数达到了 0x8004 的阈值，这是正常的网络波动。",
    "Python 的 smtplib 库也可以用来发送邮件，连接失败时会抛出 SMTPException。",
    "Milvus 是一个高性能的向量数据库，支持稠密向量和稀疏向量的混合检索。"
]

print("\n>>> 正在生成向量并插入数据...")

# 【优势 1】直接传入文本列表，返回结果包含 .dense 和 .sparse
docs_embeddings = bge_m3_ef(documents)
rows=docs_embeddings["sparse"][0].row
cols=docs_embeddings["sparse"][0].col
datas=docs_embeddings["sparse"][0].data

# print(type(docs_embeddings["sparse"]))  # CSR (Compressed Sparse Row) 格式是用来存储**2D 矩阵（多行数据） 目的：省内存，且进行矩阵运算（如批量点积）极快
# print(type(docs_embeddings["sparse"][0])) # C00 当你执行 [0] 进行切片操作时，你是在告诉 Scipy：“我只要第一行”。 对于单行数据，COO 格式简直是天赐的礼物 它的 .col 属性直接就是你要的 Token ID 它的 .data 属性直接就是你要的 权重。
#
# sparse_dict = {
#         int(k): float(v)
#         for k, v in zip(cols, datas.data)
#     }
#
# print(sparse_dict)
#
data_rows = []
for i, doc_text in enumerate(documents):

    # 1. 获取当前文档的稀疏向量对象 (可能是 coo_array（稀释矩阵的坐标格式 主要作用构建数据，不支持矩阵运算和矩阵切片） 或 csr_matrix（稀释矩阵的压缩格式 主要作用矩阵运算、切片查询）)
    sparse_row = docs_embeddings["sparse"][i]
    sparse_dict = {
        int(t_id): float(s)
        for t_id, s in zip(sparse_row.col, sparse_row.data)
    }
    data_rows.append({
        "text": doc_text,
        # 直接获取 SDK 处理好的向量
        "dense_vector": docs_embeddings["dense"][i].tolist(),
        "sparse_vector": sparse_dict # 标准字典
    })

insert_res = milvus_client.insert(collection_name=COLLECTION_NAME, data=data_rows)
print(f"成功插入 {insert_res['insert_count']} 条数据。")

### 3.4 定义混合检索 + Rerank 核心函数
- 自动计算

In [ ]:
# --- 4. 定义 混合检索+重排序 函数 ---

def search_and_rerank(query_text, top_k_retrieval=50, top_k_final=3):
    print(f"\n{'='*50}")
    print(f"查询: {query_text}")

    # --- Step A: 向量化查询 ---
    # SDK 自动生成 Dense 和 Sparse
    query_vecs = bge_m3_ef([query_text])
    dense_q = query_vecs["dense"]
    sparse_q = query_vecs["sparse"]
    print(repr(sparse_q))
    # --- Step B: 构建混合检索请求 ---
    # 1. 稠密检索请求
    req_dense = AnnSearchRequest(
        data=dense_q,
        anns_field="dense_vector",
        param={"metric_type": "COSINE", "params": {"ef": 100}},
        limit=top_k_retrieval
    )

    # 2. 稀疏检索请求
    req_sparse = AnnSearchRequest(
        data=sparse_q,
        anns_field="sparse_vector",
        param={"metric_type": "IP", "params": {"drop_ratio_search": 0.1}}, # 计算相似度时 把问题和权重做内积 算分 告诉它搜索时忽略多少微小的权重以加速
        limit=top_k_retrieval
    )

    # --- Step C: 执行 Hybrid Search (RRF 融合) ---
    res = milvus_client.hybrid_search(
        collection_name=COLLECTION_NAME,
        reqs=[req_dense, req_sparse],
        # 使用 RRFRanker 进行倒数排名融合
        ranker=RRFRanker(k=60),
        limit=top_k_retrieval,
        output_fields=["text"] # 必须返回 text 用于 Reranker
    )

    if not res or not res[0]:
        print("未找到结果。")
        return

    print(f" -> [粗排] 混合召回了 {len(res[0])} 条文档。")

    # --- Step D: Reranker 精排 ---
    print(" -> [精排] 正在使用 BGE-Reranker 重排序...")

    # 1. 【核心修复】手动提取文本
    # MilvusClient 返回的 res[0] 是一个字典列表，类似于 [{'id': 1, 'text': '...', 'score': ...}, ...]
    # 安全写法：优先取 'text'，取不到则尝试 'entity.get("text")'

    candidate_texts = []
    for hit in res[0]:
        # 如果是 MilvusClient (高层封装)，通常直接通过 key 访问
        text =  hit.get("entity", {}).get("text")
        candidate_texts.append(text)

    # 2. 调用 Reranker
    rerank_results = bge_reranker(
        query=query_text,
        documents=candidate_texts, # 传入纯文本列表
        top_k=top_k_final
    )
    # --- Step E: 展示结果 ---
    print(f"\n>>> 最终 Top {top_k_final} 结果:")
    for r in rerank_results:
        # r 是 RerankResult 对象
        print(f"   [Score: {r.score:.4f}] {r.text}")


### 3.5 测试

In [ ]:
# --- 5. 运行测试 ---

# 场景 1: 精确代码 (Sparse 优势)
search_and_rerank("Outlook 报错 0x80040115")

# 场景 2: 语义模糊 (Dense 优势)
search_and_rerank("为什么邮件发不出去？")